<a href="https://colab.research.google.com/github/Pshyam17/ML_Project_Torch/blob/main/Mamba_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Understanding Mamba and State-Space Models (SSMs)

Mamba is a novel deep learning architecture that combines the strengths of both recurrent neural networks (RNNs) and convolutional neural networks (CNNs). It achieves linear-time complexity with respect to sequence length, enabling efficient processing of very long sequences, while maintaining high performance comparable to Transformers.

At its core, Mamba relies on a **State-Space Model (SSM)**. SSMs are a powerful class of models used to describe dynamic systems, representing them through a hidden 'state' that evolves over time. They are commonly used in control theory and signal processing.

### Continuous-Time State-Space Models

In their continuous form, a linear time-invariant (LTI) State-Space Model is defined by a set of differential equations:

1.  **State Equation**: `h'(t) = A h(t) + B x(t)`
2.  **Output Equation**: `y(t) = C h(t) + D x(t)`

Where:
*   `x(t)`: Input signal at time `t`
*   `h(t)`: Hidden state vector at time `t`
*   `y(t)`: Output signal at time `t`
*   `A`: State matrix (determines how the state evolves internally)
*   `B`: Input matrix (determines how the input affects the state)
*   `C`: Output matrix (determines how the state is mapped to the output)
*   `D`: Feedthrough matrix (determines how the input directly affects the output, often zero in neural network contexts)

These matrices (`A`, `B`, `C`, `D`) are constants for LTI systems. The hidden state `h(t)` encapsulates all necessary information from past inputs to predict future outputs.

### Discretization of State-Space Models (Zero-Order Hold - ZOH)

Neural networks operate on discrete sequences, so we need to convert the continuous SSM into a discrete-time equivalent. The **Zero-Order Hold (ZOH)** method is a common way to do this. It assumes that the input `x(t)` remains constant over a small time interval `Δ` (the sampling period).

Given a time step `Δ`, the continuous matrices `A` and `B` are transformed into discrete matrices `A_bar` and `B_bar`:

1.  **Discrete State Matrix (`A_bar`)**:
    `A_bar = exp(A * Δ)`
    Where `exp` is the matrix exponential.

2.  **Discrete Input Matrix (`B_bar`)**:
    `B_bar = (A * Δ)^-1 (exp(A * Δ) - I) B` (if `A` is invertible, where `I` is the identity matrix)
    A more general form using integration is:
    `B_bar = ∫_0^Δ exp(Aτ) B dτ`

With `A_bar` and `B_bar`, the discrete-time SSM equations become:

1.  **Discrete State Recurrence**: `h_k = A_bar h_{k-1} + B_bar x_k`
2.  **Discrete Output Equation**: `y_k = C h_k + D x_k`

This recurrence relation allows us to compute the state and output at each discrete time step `k` based on the previous state and the current input. This is the core mechanism that enables RNN-like behavior in SSMs.

In [3]:
import torch

def discretize_ssm_zoh(A, B, delta):
    """
    Discretizes continuous-time SSM matrices A and B using Zero-Order Hold.
    A: (N, N) tensor
    B: (N, 1) tensor (or (N, D_in))
    delta: scalar tensor
    Returns: A_bar, B_bar
    """

    N = A.shape[0]
    D_in = B.shape[1] # Infer D_in from the shape of B

    # Construct the augmented matrix F_aug:
    # F_aug = [[A*delta, B*delta],
    #          [0 (D_in, N), 0 (D_in, D_in)]]

    # Top block: [A*delta, B*delta]
    top_block = torch.cat([A * delta, B * delta], dim=1) # Shape (N, N + D_in)

    # Bottom block: zeros of shape (D_in, N + D_in)
    bottom_block = torch.zeros(D_in, N + D_in, device=A.device)

    # Combine to form F_aug
    F = torch.cat([top_block, bottom_block], dim=0) # Shape (N + D_in, N + D_in)

    F_exp = torch.matrix_exp(F)

    # Extract A_bar and B_bar_term from the exponentiated augmented matrix
    A_bar = F_exp[:N, :N]
    B_bar_term = F_exp[:N, N:]

    return A_bar, B_bar_term

def discrete_ssm_step(h_prev, x_curr, A_bar, B_bar, C, D):
    """
    Performs one step of a discrete-time State-Space Model.
    h_prev: (N,) tensor (previous hidden state)
    x_curr: (D_in,) tensor (current input)
    A_bar: (N, N) tensor
    B_bar: (N, D_in) tensor
    C: (D_out, N) tensor
    D: (D_out, D_in) tensor
    Returns: h_curr, y_curr
    """

    # Ensure x_curr has a batch dimension or is compatible for matmul
    # If B_bar is (N, D_in) and x_curr is (D_in,), then B_bar @ x_curr will be (N,)
    # If B_bar is (N, D_in) and x_curr is (batch_size, D_in), then (B_bar @ x_curr.T).T will be (batch_size, N)
    # For this 'from scratch' example, let's assume single input vector x_curr

    # State update: h_k = A_bar h_{k-1} + B_bar x_k
    # (N,N) @ (N,) + (N, D_in) @ (D_in,)
    h_curr = torch.matmul(A_bar, h_prev) + torch.matmul(B_bar, x_curr)

    # Output update: y_k = C h_k + D x_k
    # (D_out, N) @ (N,) + (D_out, D_in) @ (D_in,)
    y_curr = torch.matmul(C, h_curr) + torch.matmul(D, x_curr)

    return h_curr, y_curr


In [4]:
# Example Usage:

# Define dimensions
N = 4  # State dimension
D_in = 2 # Input dimension
D_out = 1 # Output dimension

# Continuous-time SSM matrices (randomly initialized for demonstration)
A_cont = torch.randn(N, N)
B_cont = torch.randn(N, D_in)
C_cont = torch.randn(D_out, N)
D_cont = torch.randn(D_out, D_in)

# Time step
delta = torch.tensor(0.1)

# Discretize the SSM matrices
A_bar, B_bar = discretize_ssm_zoh(A_cont, B_cont, delta)

print(f"Continuous A shape: {A_cont.shape}")
print(f"Continuous B shape: {B_cont.shape}")
print(f"Discrete A_bar shape: {A_bar.shape}")
print(f"Discrete B_bar shape: {B_bar.shape}")

# Initial hidden state and input
h_0 = torch.zeros(N) # Initial state
x_1 = torch.randn(D_in) # First input

print(f"\nInitial hidden state h_0: {h_0}")
print(f"First input x_1: {x_1}")

# Perform one step of the discrete SSM
h_1, y_1 = discrete_ssm_step(h_0, x_1, A_bar, B_bar, C_cont, D_cont)

print(f"\nUpdated hidden state h_1: {h_1}")
print(f"Output y_1: {y_1}")

# Simulate a sequence (e.g., 5 steps)
sequence_length = 5
inputs = torch.randn(sequence_length, D_in)
hidden_states = [h_0]
outputs = []

current_h = h_0
for i in range(sequence_length):
    x_k = inputs[i]
    current_h, y_k = discrete_ssm_step(current_h, x_k, A_bar, B_bar, C_cont, D_cont)
    hidden_states.append(current_h)
    outputs.append(y_k)

print(f"\nSimulated sequence over {sequence_length} steps:")
for i in range(sequence_length):
    print(f"Step {i+1}: Input={inputs[i]}, Output={outputs[i]}")


Continuous A shape: torch.Size([4, 4])
Continuous B shape: torch.Size([4, 2])
Discrete A_bar shape: torch.Size([4, 4])
Discrete B_bar shape: torch.Size([4, 2])

Initial hidden state h_0: tensor([0., 0., 0., 0.])
First input x_1: tensor([ 0.5753, -0.9694])

Updated hidden state h_1: tensor([ 0.0512, -0.1275,  0.2890,  0.0759])
Output y_1: tensor([-0.5377])

Simulated sequence over 5 steps:
Step 1: Input=tensor([-1.0341,  1.3703]), Output=tensor([0.8811])
Step 2: Input=tensor([-1.1473,  0.4054]), Output=tensor([0.9988])
Step 3: Input=tensor([0.3368, 0.5628]), Output=tensor([0.3623])
Step 4: Input=tensor([-0.4131,  0.3567]), Output=tensor([0.7748])
Step 5: Input=tensor([ 0.4198, -0.6904]), Output=tensor([0.1505])


## The Selective State-Space Model (S-SSM)

The main innovation in Mamba, derived from the Selective State Space (S4) models, is the concept of **selectivity**. This means that the parameters of the SSM – specifically the discretization step `Δ`, and the input and output matrices `B` and `C` – are no longer fixed constants but are dynamically computed based on the input. This dynamic adaptation allows the model to filter relevant information from the input sequence and ignore irrelevant parts, providing a powerful content-aware mechanism.

In Mamba, for each input token `x_k` in a sequence, we compute a new `Δ_k`, `B_k`, and `C_k`:

*   `Δ_k = f_Δ(x_k)`: A scalar or vector representing the time-step, usually generated by a small neural network (`f_Δ`) from the input token.
*   `B_k = f_B(x_k)`: The input matrix, generated by another network (`f_B`).
*   `C_k = f_C(x_k)`: The output matrix, generated by another network (`f_C`).

The `A` matrix, however, often remains a fixed parameter, learned during training, to maintain some consistent dynamics. The `D` matrix is typically kept constant or omitted.

With these input-dependent parameters, the discrete-time SSM equations become:

1.  **Discrete State Recurrence**: `h_k = A_bar(Δ_k) h_{k-1} + B_bar(Δ_k, B_k) x_k`
2.  **Discrete Output Equation**: `y_k = C_k h_k + D x_k`

Where `A_bar(Δ_k)` and `B_bar(Δ_k, B_k)` are derived from the continuous `A` and `B_k` using the input-dependent `Δ_k`. This makes the state update process content-aware, allowing for efficient and effective long-range dependency modeling.

### Adapting Discretization for Selectivity

To implement the Selective Scan, we need to modify our `discretize_ssm_zoh` function to accept an input-dependent `B` and `Δ`. For Mamba, typically `Δ` is a scalar (or a vector of scalars for different state dimensions) and `B` is a vector derived from the input. For simplicity, we will adapt our existing `discretize_ssm_zoh` function to take a `delta_k` and `B_k` that can vary per step.

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# use the previously defined discretize_ssm_zoh and discrete_ssm_step
# Let's redefine them here for clarity with the modifications for selectivity

def discretize_ssm_zoh_selective(A, B_k, delta_k):
    """
    Discretizes continuous-time SSM matrices A and B_k using Zero-Order Hold,
    with an input-dependent delta_k and B_k.

    A: (N, N) tensor (continuous state matrix, typically fixed)
    B_k: (N, D_in) tensor (input matrix, varies per step, based on input x_k)
    delta_k: scalar tensor (time step, varies per step, based on input x_k)

    Returns: A_bar, B_bar
    """
    N = A.shape[0]
    D_in = B_k.shape[1] # Infer D_in from the shape of B_k

    # Construct the augmented matrix F_aug:
    # F_aug = [[A*delta_k, B_k*delta_k],
    #          [0 (D_in, N), 0 (D_in, D_in)]]

    # Top block: [A*delta_k, B_k*delta_k]
    top_block = torch.cat([A * delta_k, B_k * delta_k], dim=1) # Shape (N, N + D_in)

    # Bottom block: zeros of shape (D_in, N + D_in)
    bottom_block = torch.zeros(D_in, N + D_in, device=A.device)

    # Combine to form F_aug
    F = torch.cat([top_block, bottom_block], dim=0) # Shape (N + D_in, N + D_in)

    F_exp = torch.matrix_exp(F)

    # Extract A_bar and B_bar_term from the exponentiated augmented matrix
    A_bar = F_exp[:N, :N]
    B_bar_term = F_exp[:N, N:]

    return A_bar, B_bar_term

def discrete_ssm_step_selective(h_prev, x_curr, A_bar, B_bar, C_k, D):
    """
    Performs one step of a discrete-time Selective State-Space Model.

    h_prev: (N,) tensor (previous hidden state)
    x_curr: (D_in,) tensor (current input)
    A_bar: (N, N) tensor (discrete state matrix, varies with delta_k)
    B_bar: (N, D_in) tensor (discrete input matrix, varies with delta_k and B_k)
    C_k: (D_out, N) tensor (output matrix, varies per step, based on input x_k)
    D: (D_out, D_in) tensor (feedthrough matrix, typically fixed)

    Returns: h_curr, y_curr
    """
    # State update: h_k = A_bar h_{k-1} + B_bar x_k
    h_curr = torch.matmul(A_bar, h_prev) + torch.matmul(B_bar, x_curr)

    # Output update: y_k = C_k h_k + D x_k
    y_curr = torch.matmul(C_k, h_curr) + torch.matmul(D, x_curr)

    return h_curr, y_curr


### 1. Data Preparation: Character-level Language Modeling

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# For simplicity, we'll use a small, classic text dataset
# You can download a small Shakespeare text file, or generate a dummy one.
# For this example, let's create a dummy text.

text = """
'Tis but thy name that is my enemy;
Thou art thyself, though not a Montague.
What's Montague? it is nor hand, nor foot,
Nor arm, nor face, nor any other part
Belonging to a man. O, be some other name!
What's in a name? that which we call a rose
By any other word would smell as sweet;
So Romeo would, were he not Romeo call'd,
Retain that dear perfection which he owes
Without that title. Romeo, doff thy name;
And for that name which is no part of thee
Take all myself.
"""

# Create vocabulary and mappings
chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_int[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([int_to_char[i] for i in l]) # decoder: take a list of integers, output a string

print(f"Original text length: {len(text)} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"First 10 encoded characters: {encode(text[:10])}")
print(f"Decoded back: {decode(encode(text[:10]))}")

# Convert the entire text to a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\nEncoded data tensor shape: {data.shape}")


Original text length: 484 characters
Vocabulary size: 38 unique characters
First 10 encoded characters: [0, 3, 15, 25, 33, 1, 18, 35, 34, 1]
Decoded back: 
'Tis but 

Encoded data tensor shape: torch.Size([484])


In [10]:
# Split data into training and validation sets
train_ratio = 0.9
n = int(train_ratio * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")

# Function to get batches for character-level prediction
def get_batch(split, block_size, batch_size):
    data = train_data if split == 'train' else val_data
    # Generate random starting indices for the sequences in the batch
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # Stack sequences of length block_size as inputs
    x = torch.stack([data[i:i+block_size] for i in ix])
    # Stack next characters as targets
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# Example usage of get_batch
block_size = 8 # Context length for predictions
batch_size = 4 # How many independent sequences will we process in parallel?

x, y = get_batch('train', block_size, batch_size)

print(f"\nInput batch (x) shape: {x.shape}") # (batch_size, block_size)
print(f"Target batch (y) shape: {y.shape}") # (batch_size, block_size)

print("\nSample input sequence (x[0]):")
print(decode(x[0].tolist()))
print("Sample target sequence (y[0]):")
print(decode(y[0].tolist()))


Training data size: 435
Validation data size: 49

Input batch (x) shape: torch.Size([4, 8])
Target batch (y) shape: torch.Size([4, 8])

Sample input sequence (x[0]):
thyself,
Sample target sequence (y[0]):
hyself, 


In [7]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reuse the previously defined discretize_ssm_zoh_selective and discrete_ssm_step_selective
# for the core SSM operations.

class MambaBlock(nn.Module):
    def __init__(self, d_model, N, d_in=1, d_out=1, dt_min=0.0001, dt_max=0.01, block_size=8):
        super().__init__()
        self.d_model = d_model # Input dimension for the Mamba block
        self.N = N # State dimension of the SSM
        self.d_in = d_in # Input dimension for the internal SSM (often 1 for each head)
        self.d_out = d_out # Output dimension for the internal SSM (often 1 for each head)
        self.dt_min = dt_min        # minimum Δ after softplus + clamp
        self.dt_max = dt_max        # maximum Δ after softplus + clamp
        self.block_size = block_size  # K: tokens per block for B/C projection

        # Learnable continuous-time A matrix.
        # Geometric (S4D-style) per-channel init: each diagonal entry is set to
        #   A[i, i] = -exp(log(i+1) / N)  for i in range(N)
        # This spreads timescales geometrically across state dimensions so that
        # different channels capture short-, medium-, and long-range dependencies
        # independently — improving coverage on long pixel sequences.
        # Off-diagonal entries are zero; only diagonal timescales matter here.
        A_init = torch.zeros(self.N, self.N)
        for i in range(self.N):
            A_init[i, i] = -math.exp(math.log(i + 1) / self.N)
        self.A_cont = nn.Parameter(A_init)

        # D matrix (feedthrough, typically fixed or learned once, often initialized to zeros)
        self.D = nn.Parameter(torch.randn(self.d_out, self.d_in))

        # Per-channel state bias b_c (shape: [N], one entry per state dimension).
        # Added to the hidden-state recurrence as:
        #   h_t = A_bar * h_{t-1} + B_bar * x_t + b_c
        # Initialised to zero so the recurrence is unbiased at the start of
        # training; learned values prevent hidden-state collapse by allowing
        # each channel to maintain a non-zero steady-state offset.
        self.b_c = nn.Parameter(torch.zeros(self.N))

        # Linear layer to project input x to the parameters of the selective SSM:
        # delta, B, C. In Mamba, these are often derived from a shared projection.
        # For simplicity, we'll project to generate delta, B, and C independently.

        # Delta projection: maps input feature (d_model) to a scalar delta.
        # Bias initialised to a negative value so softplus maps initial outputs into
        # [dt_min, dt_max] ≈ [0.0001, 0.01], keeping step sizes small and preserving
        # long-range memory from the start of training.
        self.delta_proj = nn.Linear(self.d_model, 1)
        nn.init.constant_(
            self.delta_proj.bias,
            math.log(math.sqrt(dt_min * dt_max))  # geometric-mean bias ≈ -6.9
        )

        # B_cont projection: maps input feature (d_model) to B_k (N, d_in)
        # Here, d_in is the input dimension to the SSM per head. For simplicity, we assume d_in=1
        # for the internal SSM, and the actual input x_k becomes a scalar.
        self.B_cont_proj = nn.Linear(self.d_model, self.N * self.d_in)

        # C_cont projection: maps input feature (d_model) to C_k (d_out, N)
        self.C_cont_proj = nn.Linear(self.d_model, self.d_out * self.N)

        # Softplus ensures strict positivity; clamp applied in forward().
        self.delta_softplus = nn.Softplus()

    def forward(self, x_seq):
        # x_seq: (seq_len, d_model)

        seq_len, d_model = x_seq.shape

        # ── Block-wise B and C projections ───────────────────────────────────
        # Divide the sequence into non-overlapping blocks of size block_size K.
        # B and C for every token are derived from the mean of its block, then
        # tiled back to per-token resolution.  Using a block mean rather than a
        # per-token projection smooths out local noise and gives the gating
        # matrices a slightly longer temporal context without adding parameters.
        B_by_token = []  # one (N, d_in) tensor per token
        C_by_token = []  # one (d_out, N) tensor per token
        for blk_start in range(0, seq_len, self.block_size):
            block = x_seq[blk_start : blk_start + self.block_size]  # (<= K, d_model)
            blk_mean = block.mean(dim=0)                             # (d_model,)
            B_blk = self.B_cont_proj(blk_mean).view(self.N, self.d_in)    # (N, d_in)
            C_blk = self.C_cont_proj(blk_mean).view(self.d_out, self.N)   # (d_out, N)
            for _ in range(block.shape[0]):
                B_by_token.append(B_blk)
                C_by_token.append(C_blk)
        # ─────────────────────────────────────────────────────────────────────

        # Initial hidden state for the scan
        h = torch.zeros(self.N, device=x_seq.device)

        outputs = []

        # The 'scan' operation iterates through the sequence
        for i in range(seq_len):
            x_k = x_seq[i, :]

            # Delta: softplus for positivity, then hard-clamp to [dt_min, dt_max].
            # The tight upper bound (dt_max = 0.01) prevents large step sizes that cause
            # rapid hidden-state decay and loss of long-range context on pixel sequences
            # like Pathfinder (1 024 tokens) and Path-X (16 384 tokens).
            delta_k = self.delta_softplus(self.delta_proj(x_k))  # scalar
            delta_k = delta_k.clamp(self.dt_min, self.dt_max)

            # B_k and C_k: block-averaged projections (precomputed above).
            # Each token in the same block shares the same B and C values.
            B_k_cont = B_by_token[i]  # (N, d_in)
            C_k_cont = C_by_token[i]  # (d_out, N)

            # Discretize A_cont and B_k_cont using the dynamic delta_k
            A_bar, B_bar = discretize_ssm_zoh_selective(self.A_cont, B_k_cont, delta_k)

            # Perform one step of the discrete selective SSM
            # Note: For this simplified example, the SSM input x_curr is just a scalar '1' or part of x_k.
            # A more complete Mamba would often integrate the input x_k into B_k_cont directly.
            # Here, we'll use a simplified 'internal' input for the SSM scan to match B_k_cont.shape.
            # Let's assume the actual input `x_k` acts as the `D_in` part of `x_curr`.
            # So, `x_curr` for `discrete_ssm_step_selective` should be a tensor of shape `(d_in,)`.
            # For this example, let's just use a dummy input vector for the SSM.
            # In a real Mamba, the input to the SSM is often structured from the block input `x_k`.

            # For simplicity, we'll use a slice of the original x_k as the SSM's 'internal' input
            # to match B_k_cont's D_in dimension. This is a conceptual simplification.
            ssm_input_x_curr = x_k[:self.d_in] # Take first d_in elements of d_model as ssm input

            h, y_k = discrete_ssm_step_selective(h, ssm_input_x_curr, A_bar, B_bar, C_k_cont, self.D)
            h = h + self.b_c  # per-channel bias: h_t = A_bar*h_{t-1} + B_bar*x_t + b_c
            outputs.append(y_k)

        return torch.stack(outputs, dim=0) # Stack outputs to form a sequence (seq_len, d_out)


In [8]:
# Example Usage of the MambaBlock:

# Define dimensions
d_model = 128 # Input feature dimension to the Mamba block
N = 16        # State dimension of the SSM
d_in = 1      # Internal input dimension to the SSM (often 1, for each SSM head)
d_out = 1     # Internal output dimension from the SSM (often 1, for each SSM head)
seq_len = 10  # Length of the input sequence

# Create a dummy input sequence (batch_size=1 for simplicity)
input_sequence = torch.randn(seq_len, d_model)

# Instantiate the MambaBlock
mamba_block = MambaBlock(d_model=d_model, N=N, d_in=d_in, d_out=d_out)

# Pass the input sequence through the MambaBlock
output_sequence = mamba_block(input_sequence)

print(f"Input sequence shape: {input_sequence.shape}")
print(f"Output sequence shape: {output_sequence.shape}")
print("First 3 outputs:")
print(output_sequence[:3])


Input sequence shape: torch.Size([10, 128])
Output sequence shape: torch.Size([10, 1])
First 3 outputs:
tensor([[  5.9840],
        [-20.6541],
        [ 62.7701]], grad_fn=<SliceBackward0>)


In [6]:
# Example Usage of Selective SSM (the full Mamba block builds on this)

# Define dimensions
N = 4  # State dimension
D_in = 2 # Input dimension
D_out = 1 # Output dimension

# Continuous-time A matrix (fixed, learned parameter)
A_cont = torch.randn(N, N)

# D matrix (fixed, typically small or zero)
D_cont = torch.randn(D_out, D_in)

# Initial hidden state
h_0 = torch.zeros(N)

# Simulate a sequence (e.g., 5 steps) with selective parameters
sequence_length = 5
inputs = torch.randn(sequence_length, D_in)
hidden_states_selective = [h_0]
outputs_selective = []

current_h = h_0
for i in range(sequence_length):
    x_k = inputs[i]

    # Selective parameters generation (simplified for demonstration)
    # In a real Mamba, these would be generated by MLPs from x_k
    delta_k = torch.sigmoid(torch.randn(1)) # Example: delta_k generated from x_k, then scaled
    B_k_cont = torch.randn(N, D_in) # Example: B_k generated from x_k
    C_k_cont = torch.randn(D_out, N) # Example: C_k generated from x_k

    # Discretize for the current step with selective parameters
    A_bar, B_bar = discretize_ssm_zoh_selective(A_cont, B_k_cont, delta_k)

    # Perform one step of the discrete SSM
    current_h, y_k = discrete_ssm_step_selective(current_h, x_k, A_bar, B_bar, C_k_cont, D_cont)

    hidden_states_selective.append(current_h)
    outputs_selective.append(y_k)

print(f"\nSimulated Selective SSM sequence over {sequence_length} steps:")
for i in range(sequence_length):
    print(f"Step {i+1}: Input={inputs[i]}, Output={outputs_selective[i]}")



Simulated Selective SSM sequence over 5 steps:
Step 1: Input=tensor([1.7775, 0.4740]), Output=tensor([1.7281])
Step 2: Input=tensor([-2.1607,  0.8597]), Output=tensor([5.7177])
Step 3: Input=tensor([-1.6761,  0.7996]), Output=tensor([13.4279])
Step 4: Input=tensor([-0.1125, -0.8878]), Output=tensor([-63.5860])
Step 5: Input=tensor([-0.6231, -0.2251]), Output=tensor([-71.9178])


### 5. Label-Smoothed BCE Loss for Pathfinder

Standard cross-entropy is brittle on binary tasks like Pathfinder because the
model is pushed toward hard 0/1 probabilities, which amplifies overconfidence and
destabilises training on long sequences.  Label smoothing mixes the true target
with a uniform distribution, keeping the model calibrated:

```
loss = (1 - ε) · CE(logits, y)  +  ε · CE(logits, 0.5)
```

For binary classification `CE(logits, 0.5)` is the cross-entropy against the
uniform distribution `[0.5, 0.5]`.  `ε = 0.1` is the default.

In [ ]:
import torch.nn.functional as F


def label_smoothed_bce_loss(logits, labels, eps=0.1):
    """Label-smoothed cross-entropy for Pathfinder / Path-X binary classification.

    Combines the standard (hard) cross-entropy with a soft cross-entropy against
    the uniform distribution over classes:

        loss = (1 - eps) * CE(logits, y) + eps * CE(logits, uniform)

    For binary classification (C=2) the uniform target is [0.5, 0.5], so the
    soft term equals CE(logits, 0.5) as stated in the LRA Mamba fix spec.

    Args:
        logits : (B, num_classes) — raw model outputs, not softmaxed.
        labels : (B,) long        — ground-truth class indices.
        eps    : float            — smoothing weight in [0, 1]; default 0.1.

    Returns:
        Scalar loss tensor.
    """
    # Hard term: standard cross-entropy against ground-truth labels.
    ce_hard = F.cross_entropy(logits, labels)

    # Soft term: cross-entropy against the uniform distribution over C classes.
    # For C=2 this equals -0.5*log(p0) - 0.5*log(p1) = CE(logits, 0.5).
    log_probs = F.log_softmax(logits, dim=-1)        # (B, C)
    ce_soft   = -log_probs.mean(dim=-1).mean()       # mean over classes then batch

    return (1.0 - eps) * ce_hard + eps * ce_soft


# Quick sanity check
if __name__ == '__main__' or True:
    import torch
    _logits = torch.randn(8, 2)   # batch=8, binary
    _labels = torch.randint(0, 2, (8,))
    _loss   = label_smoothed_bce_loss(_logits, _labels, eps=0.1)
    _hard   = F.cross_entropy(_logits, _labels)
    print(f'hard CE : {_hard.item():.4f}')
    print(f'smooth  : {_loss.item():.4f}  (eps=0.1)')
    assert _loss.item() != _hard.item(), 'smoothing had no effect'
    print('label_smoothed_bce_loss OK')